In [ ]:
#LLama 70B Spanish

In [1]:
!pip install pandas
!pip install tqdm
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn
!pip install huggingface_hub
!pip install transformers
!pip3 install torch torchvision torchaudio
!pip install accelerate
!pip install -U bitsandbytes

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [1]:
# Code updates to run for Spanish dataset
import os
import json
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

##############################################################
# User-defined parameters
##############################################################
PATH_EXIST = 'EXIST2024_training_Task1.csv'
PATH_IMPORTANT_TOKENS_ES = 'important_tokens1_es.csv'  # Ensure Spanish tokens are specified

SAMPLE_RATIO = 0.1

# ##############################################################
# # Set Random Seeds for Reproducibility
# ##############################################################
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Optionally, ensure deterministic behavior in PyTorch (may impact performance)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

##############################################################
# Data Loading and Preprocessing
##############################################################
df = pd.read_csv(PATH_EXIST, encoding='utf-8')

def get_majority_vote(labels_list):
    counter = Counter(labels_list)
    counts = counter.most_common()
    if len(counts) == 1:
        return counts[0][0]
    else:
        # Tie-breaking
        top_count = counts[0][1]
        top_candidates = [c[0] for c in counts if c[1] == top_count]
        return random.choice(top_candidates)  # Reproducible due to seed

def parse_labels(labels_str):
    return eval(labels_str)

df['labels_list'] = df['labels_task1'].apply(parse_labels)
df['majority_label'] = df['labels_list'].apply(get_majority_vote)

# Map labels to binary
label_map = {'YES':1, 'NO':0}
df['label'] = df['majority_label'].map(label_map)

# Filter out any samples that are not YES/NO
df = df[df['label'].isin([0,1])]

# Separate data by language
languages = df['lang'].unique()

# Set language to English (assuming 'en' is intended; adjust if Spanish is desired)
df_lang_es = df[df['lang'] == 'es'].copy()

# Apply sample ratio to reduce dataset size for quicker tests
df_lang_es = df_lang_es.sample(frac=SAMPLE_RATIO, random_state=SEED).reset_index(drop=True)

# Optional: Verify the sampling reproducibility
print(f"Number of samples after sampling: {len(df_lang_es)}")
df_lang_es.to_csv('df_lang_es.csv')
df_lang_es.head()

/home/hmohammadi/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of samples after sampling: 366


In [ ]:
#Spansih
import os
import json
import random
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
from huggingface_hub import login

# Performance optimizations for A100
torch.backends.cuda.matmul.allow_tf32 = True

login(token='hf_REDACTED_SET_VIA_ENV_VAR')

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

##############################################################
# User-defined parameters
##############################################################
PATH_EXIST = 'EXIST2024_training_Task1.csv'
PATH_IMPORTANT_TOKENS_EN = 'important_tokens1_en.csv'
PATH_IMPORTANT_TOKENS_ES = 'important_tokens1_es.csv'
MODEL_NAME = 'meta-llama/Llama-3.3-70B-Instruct'
MAX_TOKENS = 100
LOAD_IN_8BIT = True
LANGUAGES_TO_RUN = ['es', 'en']
TEMPERATURES = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
OUTPUT_DIR_BASE = 'genai_scenarios_results_llama_70B'
scenarios = ["GenAI", "GenP", "GenXAI", "GenPXAI"]
BATCH_SIZE = 128

# Create cache directory if not exists
if not os.path.exists('./model_cache'):
    os.makedirs('./model_cache')

##############################################################
# Load Model and Tokenizer
##############################################################
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True,
    cache_dir="./model_cache"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    load_in_8bit=LOAD_IN_8BIT,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    cache_dir="./model_cache"
)

##############################################################
# Load and preprocess the dataset
##############################################################
#df = pd.read_csv(PATH_EXIST, encoding='utf-8')
df=df_lang_es

def get_majority_vote(labels_list):
    counter = Counter(labels_list)
    counts = counter.most_common()
    if len(counts) == 1:
        return counts[0][0]
    else:
        top_count = counts[0][1]
        top_candidates = [c[0] for c in counts if c[1] == top_count]
        return random.choice(top_candidates)

def parse_labels(labels_str):
    return eval(labels_str)

df['labels_list'] = df['labels_task1'].apply(parse_labels)
df['majority_label'] = df['labels_list'].apply(get_majority_vote)
label_map = {'YES':1, 'NO':0}
df['label'] = df['majority_label'].map(label_map)
df = df[df['label'].isin([0,1])]
languages = df['lang'].unique()

df['ground_truth'] = df['label'].apply(lambda x: 'yes' if x==1 else 'no')

##############################################################
# Utility functions
##############################################################
def construct_persona_description(gender, age, ethnicity, study_level, region):
    persona_description = (
        f"You are a {gender} individual, aged {age}, who identifies as {ethnicity}, "
        f"has a {study_level}, and currently resides in {region}. "
        f"You have the cultural and personal background of someone with these demographics."
    )
    return persona_description

def construct_persona_description_es(gender, age, ethnicity, study_level, region):
    persona_description = (
        f"Eres una persona {gender}, de {age} años, que se identifica como {ethnicity}, "
        f"posee un nivel de estudios {study_level}, y actualmente reside en {region}. "
        f"Tienes el trasfondo cultural y personal de alguien con estas características demográficas."
    )
    return persona_description

def bold_tokens(text, tokens_to_bold):
    tokens_lower = set([t.lower() for t in tokens_to_bold])
    words = text.split()
    new_words = []
    for w in words:
        if w.lower().strip(",.!?\"'()[]") in tokens_lower:
            new_words.append(f"**{w}**")
        else:
            new_words.append(w)
    return " ".join(new_words)

def scenario_prompt(scenario_name, text, persona_description=None, xai_tokens=None, lang='es'):
    if lang == 'en':
        base_prompt = "Read the text and answer if it is sexism or not. Answer only with 'yes' or 'no' and omit explanations.\nText: "
        persona_prefix = "Take the role of Persona ({persona_description}). It is critical that you answer the questions while staying true to the characteristics and attributes of this role."
        xai_prefix = "The parts that are **bold** need more focus."
    else:
        base_prompt = "Lee el texto y responde si es sexista o no. Responde solo con 'yes' o 'no' y omite explicaciones.\nTexto: "
        persona_prefix = "Adopta el papel de la Persona ({persona_description}). Es fundamental que respondas manteniéndote fiel a las características y atributos de este rol."
        xai_prefix = "Las partes en **negrita** necesitan más atención."

    if scenario_name == "GenAI":
        prompt = f"{base_prompt}{text}"
    elif scenario_name == "GenP":
        prompt = f"{persona_prefix}\n{base_prompt}{text}"
        prompt = prompt.format(persona_description=persona_description)
    elif scenario_name == "GenXAI":
        bolded_text = bold_tokens(text, xai_tokens) if xai_tokens else text
        prompt = f"{base_prompt}{bolded_text}\n{xai_prefix}"
    elif scenario_name == "GenPXAI":
        bolded_text = bold_tokens(text, xai_tokens) if xai_tokens else text
        prompt = f"{persona_prefix}\n{base_prompt}{bolded_text}\n{xai_prefix}"
        prompt = prompt.format(persona_description=persona_description)
    else:
        raise ValueError("Unknown scenario")
    return prompt

def scenario_accuracy(df, scenario_col):
    return accuracy_score(df['ground_truth'], df[scenario_col])

##############################################################
# Adjusted batch_inference to handle nested output
##############################################################
def batch_inference(gen_pipe, prompts, batch_size=8):
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        batch_output = gen_pipe(batch, return_full_text=False)
        # batch_output is a list of lists. For each input prompt, we get a list of candidates.
        # Typically: [[{'generated_text': '...'}], [{'generated_text': '...'}], ...]
        for candidate_list in batch_output:
            out = candidate_list[0]  # first (and usually only) generation
            reply = out['generated_text'].strip().lower()
            # Extract yes/no
            if 'yes' in reply and 'no' in reply:
                yes_idx = reply.index('yes') if 'yes' in reply else 999999
                no_idx = reply.index('no') if 'no' in reply else 999999
                final = 'yes' if yes_idx < no_idx else 'no'
            elif 'yes' in reply:
                final = 'yes'
            elif 'no' in reply:
                final = 'no'
            else:
                final = 'no'
            results.append(final)
    return results

##############################################################
# Running experiments
##############################################################
for lang_to_use in LANGUAGES_TO_RUN:
    if lang_to_use not in languages:
        print(f"Language {lang_to_use} not found in dataset. Skipping.")
        continue
    
    df_lang = df[df['lang'] == lang_to_use].copy().reset_index(drop=True)
    
    top_tokens = []
    if lang_to_use == 'en' and os.path.exists(PATH_IMPORTANT_TOKENS_EN):
        df_tokens_en = pd.read_csv(PATH_IMPORTANT_TOKENS_EN)
        TOP_N_TOKENS = 50
        top_tokens = df_tokens_en.nlargest(TOP_N_TOKENS, 'SHAP Importance')['Token'].tolist()
    elif lang_to_use == 'es' and os.path.exists(PATH_IMPORTANT_TOKENS_ES):
        df_tokens_es = pd.read_csv(PATH_IMPORTANT_TOKENS_ES)
        TOP_N_TOKENS = 50
        top_tokens = df_tokens_es.nlargest(TOP_N_TOKENS, 'SHAP Importance')['Token'].tolist()
    
    # Default demographic values if missing
    if 'gender_annotators' not in df_lang.columns:
        df_lang['gender_annotators'] = "Female" if lang_to_use == 'en' else "Femenino"
    if 'age_annotators' not in df_lang.columns:
        df_lang['age_annotators'] = "23-45"
    if 'ethnicities_annotators' not in df_lang.columns:
        df_lang['ethnicities_annotators'] = "White or Caucasian" if lang_to_use == 'en' else "Blanco o Caucásico"
    if 'study_levels_annotators' not in df_lang.columns:
        df_lang['study_levels_annotators'] = "Bachelor’s degree" if lang_to_use == 'en' else "Título de grado"
    df_lang['region'] = "Europe" if lang_to_use == 'en' else "Europa"
    
    persona_cache = {}
    
    for temp in TEMPERATURES:
        if temp == 0:
            # Greedy decoding
            gen_pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=MAX_TOKENS,
                do_sample=False
            )
        else:
            # Sampling with specified temperature
            gen_pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=MAX_TOKENS,
                temperature=temp,
                do_sample=True
            )
        
        OUTPUT_DIR = f"{OUTPUT_DIR_BASE}_{lang_to_use}_temp{temp}"
        if not os.path.exists(OUTPUT_DIR):
            os.makedirs(OUTPUT_DIR)
        
        for scenario in scenarios:
            prompts_for_scenario = []
            for idx, row in df_lang.iterrows():
                tweet_text = row['tweet']
                gender = row['gender_annotators']
                age = row['age_annotators']
                ethnicity = row['ethnicities_annotators']
                study_level = row['study_levels_annotators']
                region = row['region']

                if lang_to_use == 'en':
                    persona_key = (gender, age, ethnicity, study_level, region)
                    if persona_key not in persona_cache:
                        persona_cache[persona_key] = construct_persona_description(*persona_key)
                    persona_desc = persona_cache[persona_key]
                else:
                    persona_key = (gender, age, ethnicity, study_level, region)
                    if persona_key not in persona_cache:
                        persona_cache[persona_key] = construct_persona_description_es(*persona_key)
                    persona_desc = persona_cache[persona_key]

                prompt = scenario_prompt(scenario, tweet_text, persona_description=persona_desc, xai_tokens=top_tokens, lang=lang_to_use)
                
                # Repeat prompt 6 times for majority voting
                prompts_for_scenario.extend([prompt]*6)
            
            # Batch inference for all prompts in this scenario
            predictions = batch_inference(gen_pipe, prompts_for_scenario, batch_size=BATCH_SIZE)
            
            # Group predictions by 6 to perform majority vote
            final_preds = []
            for i in range(0, len(predictions), 6):
                block = predictions[i:i+6]
                counts = Counter(block)
                final_pred = counts.most_common(1)[0][0]
                final_preds.append(final_pred)
            
            df_lang[scenario.lower() + "_pred"] = final_preds
        
        df_lang.to_csv(os.path.join(OUTPUT_DIR, f'genai_scenarios_predictions_{lang_to_use}_temp{temp}_llama.csv'), index=False)
        
        acc_genai = scenario_accuracy(df_lang, 'genai_pred')
        acc_genp = scenario_accuracy(df_lang, 'genp_pred')
        acc_genxai = scenario_accuracy(df_lang, 'genxai_pred')
        acc_genpxai = scenario_accuracy(df_lang, 'genpxai_pred')
        
        print(f"Accuracy for {lang_to_use} at temp {temp}:")
        print("GenAI:", acc_genai)
        print("GenP:", acc_genp)
        print("GenXAI:", acc_genxai)
        print("GenPXAI:", acc_genpxai)
        
        labels = ['yes', 'no']
        scenarios_pred_cols = ['genai_pred', 'genp_pred', 'genxai_pred', 'genpxai_pred']
        
        for scenario_col in scenarios_pred_cols:
            print(f"Classification Report for {scenario_col} ({lang_to_use}, temp={temp}):")
            print(classification_report(df_lang['ground_truth'], df_lang[scenario_col], labels=labels))
            cm = confusion_matrix(df_lang['ground_truth'], df_lang[scenario_col], labels=labels)
            print(f"Confusion Matrix - {scenario_col} ({lang_to_use}, temp={temp}):")
            print(cm)
        
        summary = {
            'accuracy': {
                'GenAI': float(acc_genai),
                'GenP': float(acc_genp),
                'GenXAI': float(acc_genxai),
                'GenPXAI': float(acc_genpxai)
            },
            'num_samples': len(df_lang),
            'temperature': temp
        }
        
        with open(os.path.join(OUTPUT_DIR, f'scenario_summary_{lang_to_use}_temp{temp}_llama.json'), 'w') as f:
            json.dump(summary, f, indent=4)

print("All done for all languages and temperatures. Results saved in separate directories.")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 30/30 [00:25<00:00,  1.17it/s]
Device set to use cuda:0
/home/hmohammadi/.local/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/hmohammadi/.local/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
You seem to be using the pipelines sequentially on GPU. In order to maximize 